# Hugging Face Translation Layer

This notebook translates the from-scratch curriculum into the main PyTorch and Hugging Face abstractions you will see in production-adjacent code. The goal is not to hide the earlier work. It is to show which manual pieces turn into library objects, and which responsibilities still stay with the engineer.

## 1. From text file to Hugging Face datasets

Our package starts from plain text, slices it into training examples, and returns tensors. The `datasets` library adds schema-aware tables, cached transforms, and split management. That removes glue code around reading, filtering, and shuffling text, but it does not decide whether the text is representative, clean, or aligned with the objective you care about.

In [ ]:
from datasets import Dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from transformers import GPT2Config, GPT2LMHeadModel

dataset = Dataset.from_dict({"text": ["attention reads earlier tokens", "loss trains logits"]})
config = GPT2Config(vocab_size=128, n_positions=32, n_embd=64, n_layer=2, n_head=4)
model = GPT2LMHeadModel(config)
dataset, Tokenizer(BPE()), model.num_parameters()

## 2. From `CharTokenizer` to tokenizers BPE

`CharTokenizer` makes the token-ID mapping explicit, which is exactly why it is valuable early in the curriculum. Hugging Face `tokenizers` generalizes that idea into a configurable pipeline: normalization, pre-tokenization, model training, and decoding. Byte-pair encoding reduces sequence length relative to character tokenization, but you still need to reason about vocabulary coverage, unknown tokens, and how token boundaries change the effective context window.

In [ ]:
from llm_from_scratch.configs import ModelConfig
from llm_from_scratch.model import TinyGPT, count_parameters
from llm_from_scratch.tokenization import CharTokenizer, train_bpe_tokenizer

sample_text = "attention reads earlier tokens\nloss trains logits"
char_tokenizer = CharTokenizer.from_text(sample_text)
bpe_tokenizer = train_bpe_tokenizer(sample_text.splitlines(), vocab_size=32)
scratch_config = ModelConfig(vocab_size=char_tokenizer.vocab_size, block_size=32, n_embd=64, n_head=4, n_layer=2)
scratch_model = TinyGPT(scratch_config)
{
    "char_vocab_size": char_tokenizer.vocab_size,
    "char_ids": char_tokenizer.encode("loss"),
    "bpe_vocab_size": bpe_tokenizer.get_vocab_size(),
    "tinygpt_parameters": count_parameters(scratch_model),
}

## 3. From `ModelConfig` to transformer config objects

`ModelConfig` stores the shape decisions that define our decoder: vocabulary size, context length, embedding width, head count, layer count, dropout, and bias usage. In Transformers, config objects such as `GPT2Config` play the same role. The names vary a bit across model families, but the job is the same: declare architecture and generation-relevant defaults separately from the checkpoint weights.

## 4. From `TinyGPT.forward` to `AutoModelForCausalLM`

`TinyGPT.forward` takes token IDs, applies embeddings, attention blocks, and an LM head, then optionally computes shifted cross-entropy loss. `AutoModelForCausalLM` wraps that same causal-language-model contract behind model-family-specific implementations. The abstraction saves boilerplate, but it still assumes you understand batch shape, sequence length, attention masks, tied embeddings, and what the logits mean.

## 5. From handmade training loop to `Trainer`

Our training loop made every step visible: batching, forward pass, loss, backward pass, optimizer step, evaluation intervals, and checkpoint saves. `Trainer` collects those concerns behind a higher-level API and integrates logging, callbacks, mixed precision, and evaluation hooks. That is useful once you understand the pieces, because you can tell when a default is acceptable and when you need to override it.

## 6. From handmade generate loop to `generate()`

The from-scratch `generate` function exposes autoregressive decoding directly: crop to context window, run the model, sample the next token, append, repeat. `GenerationMixin.generate()` packages the same loop with strategies such as greedy decoding, temperature sampling, beam search, and stopping criteria. It is still the same causal process, so the earlier notebook on sampling remains the conceptual reference.

## 7. What abstractions hide and what they cannot remove

Modern libraries mostly remove wiring: data plumbing, model registry code, checkpoint serialization, and training harness boilerplate. They cannot remove the need to understand token shapes, causal masking, target shifting, optimizer behavior, device placement, sampling tradeoffs, or whether an evaluation result is actually meaningful. If those foundations are weak, a polished abstraction only makes mistakes faster.

## Abstraction Bridge

| From-scratch concept | PyTorch or Hugging Face abstraction | What still matters |
| --- | --- | --- |
| Token list from a text file | `Dataset` column or tensor dataset | Example boundaries, filtering, and split quality |
| `CharTokenizer` | `tokenizers.Tokenizer` or `AutoTokenizer` | Vocabulary design, context length, unknown-token behavior |
| `ModelConfig` | `GPT2Config` or another Transformers config | Hidden sizes, attention heads, position limits |
| `TinyGPT` | `AutoModelForCausalLM`-style model | Logit shapes, masks, weight tying, loss targets |
| Handmade training loop | `Trainer` or custom Accelerate loop | Optimizer schedule, evaluation cadence, checkpoint criteria |
| Handmade generation loop | `GenerationMixin.generate()` | Sampling strategy, stop conditions, context truncation |
| Checkpoint dictionary | `save_pretrained()` and `from_pretrained()` | Which weights, tokenizer files, and configs must travel together |